# 07주차 · 감성 분석
## 영화 리뷰에서 상품 리뷰까지
**학생용 · 실습 A/B/C TODO / 응답란 포함**
**자연언어처리 | 충북대학교 | 임화정**

5주차의 TF-IDF와 6주차의 NB/LR를 하나의 감성 분석 파이프라인으로 연결합니다.

### 학습목표
1. 규칙 기반과 지도학습 기반 감성 분석의 차이를 설명한다.
2. 한국어 부정 표현과 혼합 감성이 어려운 이유를 설명한다.
3. 데이터 분할 → 학습 → 검증 → 최종 평가를 누수 없이 수행한다.
4. macro-F1, 혼동행렬, 오분류 문장으로 모델의 한계를 분석한다.

### 실행 안내
Google Colab 또는 Jupyter에서 **위에서부터 순서대로 실행**합니다. GPU·JVM·외부 데이터 다운로드는 필요하지 않습니다. Python 3.10 이상, numpy·pandas·scikit-learn을 사용합니다. 아래 설치 셀은 필요할 때만 직접 실행합니다.

```python
%pip install numpy pandas scikit-learn
```

**데이터 주의:** 아래 리뷰는 이 수업을 위해 작성한 교육용 문장입니다. NSMC 등 실제 서비스 데이터가 아니며, 작은 표본의 성능을 실무 성능으로 일반화할 수 없습니다.

## 0. 환경과 재현성
시드를 고정하더라도 라이브러리 버전에 따라 일부 결과가 달라질 수 있습니다. 버전을 함께 기록하세요.

In [1]:
import re
import sys
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
LABELS = [0, 1]  # 0: 부정, 1: 긍정
LABEL_NAMES = ['부정', '긍정']
print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__, '| sklearn:', sklearn.__version__)


Python: 3.13.18
pandas: 2.2.3 | sklearn: 1.6.1


## 1. 데이터 이해와 전처리
문서 수준의 이진 극성 분류입니다. **중립·혼합 라벨은 학습하지 않습니다.** 혼합 문장의 라벨은 이 수업에서 작성자가 의도한 전체 평가를 기준으로 부여했습니다. 다른 평가자의 판단과 다를 수 있습니다.

최소 전처리만 적용합니다. `안`, `못`, `않다`, `없다`, `ㅋㅋ`, `ㅠㅠ`는 감성 단서가 될 수 있으므로 무조건 삭제하지 않습니다. 공백 단위 벡터화는 형태소 분석과 다릅니다. 이를 간단한 baseline으로 먼저 측정합니다.

In [2]:
positive_reviews = [
 '영화가 정말 재미있고 감동적이었다',
 '배우들의 연기가 훌륭해서 몰입했다',
 '스토리가 탄탄하고 결말이 좋았다',
 '영상이 아름답고 음악도 훌륭했다',
 '다시 보고 싶은 재미있는 영화다',
 '기대보다 훨씬 재미있어서 만족했다',
 '웃음과 감동을 함께 주는 좋은 작품이다',
 '연출이 섬세하고 배우의 연기가 자연스럽다',
 '시간이 아깝지 않은 훌륭한 영화다',
 '가족과 함께 즐겁게 볼 수 있었다',
 '주인공의 이야기가 마음에 오래 남았다',
 '긴장감이 넘치고 마지막까지 재미있었다',
 '지루할 틈 없이 흥미로운 장면이 이어졌다',
 '음악과 장면이 잘 어울려서 감동했다',
 '배우의 표정이 생생해서 마음이 움직였다',
 '친구에게 추천하고 싶은 멋진 작품이다',
 '결말이 따뜻해서 기분이 좋아졌다',
 '대사가 재치 있고 웃긴 장면이 많았다',
 '액션이 시원하고 전개가 빠르다',
 '오랜만에 만족스러운 영화를 만났다',
 '평이 나빴지만 나는 재미있게 봤다',
 '초반은 느렸지만 결말은 감동적이었다',
 '완벽하지는 않아도 충분히 볼 만했다',
 '유치할 줄 알았는데 생각보다 좋았다',
 '배우가 좋아서 봤는데 내용도 재미있었다',
 '재미없다는 말에 동의하지 않는다 나는 만족했다',
 '돈이 아깝지 않았다 다시 보고 싶다',
 '기대 안 했는데 정말 재미있다 ㅋㅋ',
 '크게 웃지는 않았지만 잔잔한 감동이 좋았다',
 '긴 영화인데도 지루하지 않고 몰입했다'
]
negative_reviews = [
 '영화가 너무 지루하고 재미없었다',
 '배우의 연기가 어색해서 몰입하기 어려웠다',
 '스토리가 엉성하고 결말이 실망스러웠다',
 '영상은 화려하지만 내용은 별로였다',
 '다시는 보고 싶지 않은 최악의 영화다',
 '기대했는데 너무 실망해서 화가 났다',
 '웃음도 감동도 없는 지루한 작품이다',
 '연출이 산만하고 배우의 연기가 부자연스럽다',
 '시간이 아까운 영화라 추천하지 않는다',
 '가족과 봤는데 모두 재미없다고 했다',
 '주인공의 이야기가 이해되지 않고 답답했다',
 '긴장감이 없고 마지막까지 지루했다',
 '흥미로운 장면이 없이 같은 내용만 반복한다',
 '음악이 너무 커서 대사에 집중하지 못했다',
 '배우의 표정이 과장되어서 보기 불편했다',
 '친구에게 추천하고 싶지 않은 작품이다',
 '결말이 허무해서 기분이 나빠졌다',
 '대사가 뻔하고 웃긴 장면도 없었다',
 '액션은 많지만 전개가 산만하다',
 '오랜만에 정말 실망스러운 영화를 만났다',
 '평이 좋았지만 나는 재미없게 봤다',
 '초반은 좋았지만 결말이 너무 별로였다',
 '배우는 훌륭하지만 영화 전체는 지루했다',
 '재미있을 줄 알았는데 생각보다 나빴다',
 '배우가 좋아서 봤는데 내용은 실망이었다',
 '재미있다는 말에 동의하지 않는다 나는 실망했다',
 '돈이 아까웠다 다시 보고 싶지 않다',
 '기대 많이 했는데 정말 재미없다 ㅠㅠ',
 '웃기는 장면도 있지만 전체적으로 불쾌했다',
 '짧은 영화인데도 지루해서 끝까지 못 봤다'
]
raw_df = pd.DataFrame({
    'text': positive_reviews + negative_reviews,
    'label': [1] * len(positive_reviews) + [0] * len(negative_reviews)
})

def normalize_text(text):
    return re.sub(r'\s+', ' ', str(text)).strip() # 반복된 공백만 공백 하나로 바꾸고, 맨앞과 맨뒤의 공백만 제거.

df = raw_df.dropna(subset=['text', 'label']).copy() # text, label에 결측치 없는 행만 새로운 데이터프레임으로 복사.
df['text'] = df['text'].map(normalize_text) # text에 대해 간단한 정규화 진행
df = df[df['text'].str.len() > 0].copy() # text의 길이가 0보다 큰 행만 새로운 데이터프레임으로 복사.

# 실제 데이터는 같은 텍스트의 상충 라벨을 별도로 검토해야 한다.
assert df.groupby('text')['label'].nunique().max() == 1, '상충 라벨을 먼저 검토하세요.' # 한 텍스트에 대해 라벨이 2가지 달린 경우 없는지 체크(조건식이 False라면 AssertionError의 메시지가 출력).
assert set(df['label']) == {0, 1} #

df = df.drop_duplicates(subset=['text']).reset_index(drop=True) # 동일한 텍스트가 있다면 제거. 인덱스 다시 정렬.
df['review_id'] = np.arange(len(df)) # 0부터 len(df)-1번까지

print('정제 전/후:', len(raw_df), len(df))
display(df.groupby('label').size().rename(index={0:'부정', 1:'긍정'}).to_frame('count')) # label 기준으로 그룹화하고, 고유값별 행 개수를 센 다음에, 인덱스를 숫자에서 단어로 바꾼 뒤, 개수 열의 이름을 'count'로 지정해 df로 변환.
display(df.sample(6, random_state=SEED)) # 랜덤하게 6개의 행 출력


정제 전/후: 60 60


,count
label,
부정,30
긍정,30


,text,label,review_id
0,영화가 정말 재미있고 감동적이었다,1,0
5,기대보다 훨씬 재미있어서 만족했다,1,5
36,웃음도 감동도 없는 지루한 작품이다,0,36
45,친구에게 추천하고 싶지 않은 작품이다,0,45
13,음악과 장면이 잘 어울려서 감동했다,1,13
54,배우가 좋아서 봤는데 내용은 실망이었다,0,54


### 생각해보기 ①
`재미없다는 말에 동의하지 않는다`는 왜 부정이라고 단정하기 어려울까요? 같은 리뷰나 같은 작성자의 문장이 train/test에 동시에 들어가면 어떤 문제가 생길까요?


- '재미없다는 말'이라는 부정 표현에 '동의하지 않는다'라는 부정이 붙어, 긍정의 의미가 되었기 때문입니다.
- 같은 리뷰가 train/test에 동시에 들어가면 일반화 성능에 비해서 테스트로 평가했을 때 상대적으로 높게 나옵니다.

## 2. 규칙 기반 baseline
아래 사전은 **교육용 간이 사전**이며 KNU 등 공개 감성사전을 복제한 것이 아닙니다. 어근 문자열 포함 여부를 한 번만 검사하므로 형태소 분석, 빈도, 반어, 부정어 범위를 제대로 처리하지 못합니다. 점수 0은 `판단보류`로 출력하며 실제 중립 정답과 동일하다는 뜻이 아닙니다.

In [3]:
LEXICON = {'재미있': 2, '감동': 2, '훌륭': 2, '좋': 1, '만족': 1,
           '재미없': -2, '지루': -2, '실망': -2, '별로': -1, '아깝': -1, '최악': -2}

def lexicon_score(text):
    return sum(weight for term, weight in LEXICON.items() if term in text) # 감정사전의 모든 용어 중에서, text에 등장한 용어의 가중치를 더함.

def score_to_label(score):
    return '긍정' if score > 0 else ('부정' if score < 0 else '판단보류')

rule_examples = ['정말 재미있다', '정말 재미있지 않다', '돈이 아깝지 않았다',
                 '연기는 훌륭하지만 이야기는 지루하다', '영화는 어제 개봉했다']
rule_table = pd.DataFrame({'text': rule_examples})
rule_table['score'] = rule_table['text'].map(lexicon_score)
rule_table['prediction'] = rule_table['score'].map(score_to_label)
display(rule_table)


,text,score,prediction
0,정말 재미있다,2,긍정
1,정말 재미있지 않다,2,긍정
2,돈이 아깝지 않았다,-1,부정
3,연기는 훌륭하지만 이야기는 지루하다,0,판단보류
4,영화는 어제 개봉했다,0,판단보류


- 부정표현 '않'을 포착하지 못했습니다.

### 실습 A · 부정 표현 규칙 추가
`재미있지 않다`를 부정 점수로 처리하는 **구문 한정 규칙**을 추가하세요. 이 한 가지 수정으로 한국어 부정어 전체를 처리했다고 주장해서는 안 됩니다.
- 학생용 기본 셀은 원래 점수를 반환합니다. TODO 부분을 수정하세요.
- 나머지 문장의 예상 결과와 부작용도 설명하세요.

In [5]:
def improved_rule_score(text):
    score = lexicon_score(text)
    # TODO A: '재미있지 않' 구문을 발견하면 +2를 -2로 바꿔보세요.
    if '않' in text:
      score = score * (-1)

    return score

rule_table['improved_score'] = rule_table['text'].map(improved_rule_score)
rule_table['improved_prediction'] = rule_table['improved_score'].map(score_to_label)
display(rule_table)
print('수정 완료 확인:', improved_rule_score('정말 재미있지 않다') < 0)


,text,score,prediction,improved_score,improved_prediction
0,정말 재미있다,2,긍정,2,긍정
1,정말 재미있지 않다,2,긍정,-2,부정
2,돈이 아깝지 않았다,-1,부정,1,긍정
3,연기는 훌륭하지만 이야기는 지루하다,0,판단보류,0,판단보류
4,영화는 어제 개봉했다,0,판단보류,0,판단보류


수정 완료 확인: True


## 3. 텍스트 상태로 train / validation / test 분할
60% / 20% / 20%, 층화 분할입니다. 분할한 뒤 vectorizer를 train에만 fit합니다. `stratify`는 클래스 비율을 유지하는 방법이지 클래스 불균형 자체를 해결하는 방법은 아닙니다.

**test 봉인:** 규칙/모델/전처리 변경은 validation으로 결정합니다. 최종 test 결과를 본 뒤 개선하면 새로운 독립 평가 데이터가 필요합니다.

In [6]:
train_df, heldout_df = train_test_split(df, test_size=0.4, stratify=df['label'], random_state=SEED) # 6:4
valid_df, test_df = train_test_split(heldout_df, test_size=0.5, stratify=heldout_df['label'], random_state=SEED) # 2:2

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for a, b in [(train_df, valid_df), (train_df, test_df), (valid_df, test_df)]: # 2개씩 짝지어서 3번 비교
    assert set(a.review_id).isdisjoint(set(b.review_id)) # 리뷰 아이디가 겹치지 않는지
    assert set(a.text).isdisjoint(set(b.text)) # 텍스트가 겹치지 않는지

# 각 데이터셋의 label별 데이터 개수를 구하고, 인덱스 기준으로 정렬
display(pd.DataFrame({name: part_df.label.value_counts().sort_index()
                     for name, part_df in [('train', train_df), ('validation', valid_df), ('test', test_df)]}))


,train,validation,test
label,,,
0,18,6,6
1,18,6,6


## 4. 공정한 NB / LR 비교
공백 단위 unigram + bigram TF-IDF 조건을 두 모델에 동일하게 적용합니다. bigram은 국소 어순을 보존하지만 문장 전체 의미나 부정 범위를 이해하지는 못합니다. 단순 baseline을 만들기 위해 형태소 분석은 사용하지 않습니다.

### 실습 B · Pipeline 만들기
학생용에서는 NB 완성 예제를 참고해 LR Pipeline을 구현하세요. 기본값은 NB 복제이므로 수정 전 비교는 두 개의 동일 모델입니다.

In [7]:
def make_pipeline(classifier):
    return Pipeline([
        ('tfidf', TfidfVectorizer(tokenizer=str.split, token_pattern=None,
                                 lowercase=False, ngram_range=(1, 2), min_df=1)),
        ('clf', classifier)
    ])

models = {
    'NB': make_pipeline(MultinomialNB(alpha=1.0)),
    # TODO B: 아래 NB를 LogisticRegression으로 바꾸세요. C=1.0, max_iter=1000, random_state=SEED
    'LR': make_pipeline(LogisticRegression(C=1.0, max_iter=1000, random_state=SEED))
}

def evaluate(y_true, y_pred):
    return {'accuracy': accuracy_score(y_true, y_pred),
            'precision_positive': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
            'recall_positive': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
            'f1_positive': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
            'macro_f1': f1_score(y_true, y_pred, average='macro', labels=LABELS, zero_division=0)}

validation_predictions = {}
validation_rows = []
for name, model in models.items():
    model.fit(train_df.text, train_df.label) # train_set으로 학습
    pred = model.predict(valid_df.text) # valid_set으로 예측
    validation_predictions[name] = pred
    validation_rows.append({'model': name, **evaluate(valid_df.label, pred)}) # 성능 확인(언패킹 이용해 평가 데이터를 리스트에 추가)
validation_results = pd.DataFrame(validation_rows).set_index('model')
display(validation_results.round(3))
print('LR 수정 완료:', isinstance(models['LR'].named_steps['clf'], LogisticRegression)) # models['LR']에 배정된 모델의 타입이 LogisticRegression이 맞는지 확인

,accuracy,precision_positive,recall_positive,f1_positive,macro_f1
model,,,,,
NB,0.417,0.400,0.333,0.364,0.413
LR,0.583,0.571,0.667,0.615,0.580


LR 수정 완료: True


### 규칙 baseline도 validation에서 확인
점수 0의 판단보류는 제외한 **coverage와 선택적 정확도**를 따로 보고합니다. 전부를 예측하는 ML 정확도와 선택적 정확도는 직접 비교할 수 없습니다.

In [ ]:
rule_scores = valid_df.text.map(lexicon_score)
covered = rule_scores.ne(0) # 0과 다른지
rule_pred = (rule_scores[covered] > 0).astype(int)
print('판단 가능 비율(coverage):', round(float(covered.mean()), 3))
if covered.any():
    print('판단 가능한 문장에서만 정확도:', round(accuracy_score(valid_df.loc[covered, 'label'], rule_pred), 3))
print('판단보류 수:', int((~covered).sum()))


## 5. Validation 오류 분석과 특징 해석
**test는 아직 보지 않습니다.** 오류에는 라벨 모호성, 부정어 범위, 혼합 감성, 미등록 표현, 희소한 어휘 등이 섞일 수 있습니다.

### 실습 C · 오분류만 추출
학생용에서는 TODO 필터를 수정하세요. 빈 표가 나오면 해당 분할에서 오류가 없다는 뜻이지 모델이 완벽하다는 뜻은 아닙니다.

In [ ]:
best_name = validation_results.sort_values('macro_f1', ascending=False, kind='stable').index[0]
selected_model = models[best_name]
valid_analysis = valid_df[['review_id', 'text', 'label']].copy()
valid_analysis['prediction'] = validation_predictions[best_name]
valid_analysis['error_type'] = ''

# TODO C: label과 prediction이 다른 행만 추출하세요.
errors = valid_analysis.copy()  # 현재는 모든 행이 나옵니다.이 부분 수정하세요!

print('validation 선택 모델:', best_name, '(동률이면 사전 순서 NB 우선)')
display(errors)


### LR 가중치 읽기
클래스 1=긍정인 이진 LR에서 양수 계수는 긍정 방향, 음수는 부정 방향의 근거입니다. 계수는 인과관계가 아니며 훈련 데이터와 정규화 조건에 의존합니다.

In [ ]:
lr_classifier = models['LR'].named_steps['clf']
if isinstance(lr_classifier, LogisticRegression):
    assert list(lr_classifier.classes_) == [0, 1]
    weights = pd.DataFrame({'feature': models['LR'].named_steps['tfidf'].get_feature_names_out(),
                            'weight': lr_classifier.coef_[0]})
    display(weights.nsmallest(8, 'weight'))
    display(weights.nlargest(8, 'weight'))
else:
    print('실습 B에서 LR를 완성하면 계수 해석 표를 볼 수 있습니다.')


### 선택 확장 · 전처리와 n-gram 실험
최종 test 실행 **이전**에만 수행하세요. `ngram_range=(1,1)`과 `(1,2)`를 동일 train/validation에서 비교하고, 필요하면 `C` 또는 `alpha`를 바꿉니다. 비교를 많이 반복하면 validation에도 과적합할 수 있습니다.

형태소 분석 확장에서는 Kiwi/KoNLPy를 사용할 수 있지만 기본 실습의 필수 의존성은 아닙니다. 명사만 남기는 전처리는 형용사·동사·부정 표현을 잃을 수 있습니다. 형태소 분석기 선택, 설치, 품사 필터, 부정어 보존 정책을 기록하세요.

## 6. 선택 확정 → train+validation 재학습 → test 단 한 번
선택 기준은 **validation macro-F1 최대**, 동률이면 NB 우선입니다. 단 한 분할·소규모 데이터로 얻은 결과이므로 우월성을 입증하지 않습니다.

**주의:** 아래 셀에서 최종 평가를 마칩니다. 결과를 본 뒤 모델을 수정하고 같은 test로 재평가하지 마세요. 학생용은 실습 B를 완료한 뒤 실행하세요.

In [ ]:
from sklearn.base import clone
# TODO B 완료 여부를 먼저 확인하세요. 미완성이면 평가를 보류합니다.
if not isinstance(models['LR'].named_steps['clf'], LogisticRegression):
    print('실습 B에서 LR를 완성한 뒤 이 셀부터 다시 실행하세요. test는 아직 평가하지 않았습니다.')
elif globals().get('TEST_EVALUATED', False):
    print('이미 test 평가를 완료했습니다. 저장된 결과를 참고하세요.')
else:
    final_model = clone(models[best_name])
    development_df = pd.concat([train_df, valid_df], ignore_index=True)
    final_model.fit(development_df.text, development_df.label)
    test_pred = final_model.predict(test_df.text)
    test_metrics = evaluate(test_df.label, test_pred)
    test_cm = confusion_matrix(test_df.label, test_pred, labels=LABELS)
    TEST_EVALUATED = True
    display(pd.DataFrame([test_metrics], index=[best_name]).round(3))
    print(classification_report(test_df.label, test_pred, labels=LABELS,
                                target_names=LABEL_NAMES, zero_division=0))
    display(pd.DataFrame(test_cm, index=['실제 부정', '실제 긍정'], columns=['예측 부정', '예측 긍정']))
    print('행=실제, 열=예측. 긍정 기준: 좌상 TN, 우상 FP, 좌하 FN, 우하 TP')


### 생각해보기 ②
1. 긍정 확률이 0.51인 리뷰를 중립이라고 부를 수 있나요?
2. macro-F1과 긍정 클래스 F1은 무엇이 다른가요?
3. 부정 리뷰를 놓치는 비용이 크다면 어떤 재현율을 봐야 하나요?


## 7. 새로운 리뷰와 상품 도메인 탐색
아래 문장은 교육용 작성 예시이며 **정답 라벨이 없는 탐색 데이터**입니다. 상품 리뷰에 대한 정확도를 계산하거나 영화 모델의 도메인 이전 성능을 입증하지 않습니다. 정량 비교에는 독립적으로 라벨링한 상품 평가셋이 필요합니다.

확률 열은 `classes_` 순서를 확인해 매핑합니다. 0.5에 가깝다는 이유만으로 중립으로 재라벨링하지 않습니다.

In [ ]:
new_reviews = pd.DataFrame({'text': [
    '연기는 좋았지만 이야기가 너무 지루했다',
    '기대하지 않았는데 영화가 재미있었다',
    '정말 대단하네요 두 시간을 이렇게 낭비하다니',
    '배송은 빠르지만 제품이 고장났다',
    '가격이 비싸지만 성능이 좋아서 만족한다',
    '배터리가 오래가고 화면이 훌륭하다',
    '제품이 가볍지만 너무 쉽게 깨졌다'
], 'domain': ['영화'] * 3 + ['상품'] * 4})
if globals().get('TEST_EVALUATED', False):
    new_reviews['prediction'] = final_model.predict(new_reviews.text)
    probabilities = final_model.predict_proba(new_reviews.text)
    for col, label in enumerate(final_model.classes_):
        new_reviews['p_' + LABEL_NAMES[int(label)]] = probabilities[:, col]
    display(new_reviews)
else:
    print('실습 B와 최종 평가를 완료한 뒤 실행하세요.')


## 8. 실습 D · 해석 보고서 / 중간 프로젝트 준비
아래 마크다운 응답을 직접 작성하세요.

1. 사전 규칙이 틀리는 문장 두 개와 이유를 제시하세요.
2. validation의 NB/LR accuracy와 macro-F1을 기록하고 선택 근거를 쓰세요.
3. 오분류 두 개를 유형화하세요. 오류가 두 개보다 적으면 그 사실을 쓰고 모호한 문장을 추가 분석하세요.
4. test 혼동행렬에서 FP/FN 문서 수와 의미를 설명하세요.
5. 상품 리뷰 두 개의 예측을 읽고 도메인 이동의 위험을 설명하세요.
6. 형태소 분석 또는 n-gram 실험을 **validation에만** 수행한 개선 계획을 쓰세요.

### 제출 체크리스트
- [ ] 실습 A/B/C TODO를 완료했다.
- [ ] 위에서부터 실행할 수 있고, 실행 결과를 저장했다.
- [ ] 데이터 출처(교육용 자체 작성)·전처리·시드·버전을 기록했다.
- [ ] train/validation/test 역할을 구분하고 test를 튜닝에 쓰지 않았다.
- [ ] 지표와 오류 분석을 함께 기록했다.
- [ ] 8주차 프로젝트의 문제, 데이터, baseline, 평가 계획을 정했다.

**제출안(수업용 제안):** `학번_이름_07주차.ipynb`와 프로젝트 README를 GitHub에 정리하고 LMS에 노트북을 제출합니다.

### 응답란
- 규칙 오류:
- validation 결과와 선택 이유:
- 오류 유형:
- test FP/FN 해석:
- 도메인 이동:
- 개선 계획:


## 참고 자료
- 제공 자료: 「Part2 전통 머신러닝 5–7주차 강의안」 7주차(18–22쪽)
- 제공 자료: 「06 텍스트 분류」 및 「자연언어처리 강의계획서」
- [scikit-learn Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [scikit-learn 데이터 누수 주의사항](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)

**원자료 보완:** 전체 데이터 벡터화 후 분할하는 예시 대신 분할 후 Pipeline 학습을 사용. 실제 분석에는 형태소 분석이 항상 필수인 것은 아니므로 공백 baseline과 선택 확장을 구분. 데이터·사전·추가 예시는 본 수업용으로 새로 작성.